# UATD · RF-DETR on Kaggle (resumable)

Thin runner: all logic lives in the repo. **Every stage can be re-run safely**: finished stages are skipped, and training resumes from its last checkpoint.

**Before running**
1. Settings → Accelerator **GPU T4**, Internet **on**.
2. Add your input dataset `mdazmineaminmormo/uatd-dataset` (Input → + Add Input → Your Work). The notebook finds it automatically.
3. Settings → **Persistence → Files only**. `/kaggle/working` (checkpoints, predictions, stage markers) then survives a disconnect or restart.

**If the runtime disconnects:** reopen the notebook and click *Run All*. Done stages are skipped, and training continues from `last.ckpt`.

**Unattended (recommended for long runs):** *Save Version → Save & Run All (Commit)*. Training stops cleanly before the 12 h limit (`TRAIN_HOURS`). To continue in a new version, add the previous version's **output** as an input and set `PREV_OUTPUT` below. Its `runs/` folder is copied back, and training resumes.

In [ ]:
# ---- configuration ----
import os, time, json, shutil, subprocess
from pathlib import Path
SESSION_START = time.time()

REPO   = 'https://github.com/azmineamin/uatd-sonar-detection.git'
BRANCH = 'main'
RAW    = None   # folder containing UATD_Training/, UATD_Test_1/, UATD_Test_2/; None = search /kaggle/input
WORK   = Path('/kaggle/working')
CODE   = WORK / 'uatd'                  # repo checkout
RUNS   = WORK / 'runs' / 'rfdetr'       # checkpoints + metrics (persisted)
STATE  = WORK / 'state'                 # stage markers (persisted)
PREV_OUTPUT = ''                        # e.g. '/kaggle/input/<this-notebook>' to resume from a previous version

SMOKE  = False          # True: 6 images/split, 1 epoch, to check the setup end to end
MODEL  = 'small'        # nano | small | medium | base | large
EPOCHS, BATCH, GRAD_ACCUM, SEED = 100, 8, 2, 0
RESOLUTION  = None      # None = model default (small: 512)
TRAIN_HOURS = 11.0      # wall-clock budget per session incl. setup; Kaggle limit is 12 h
PROCESSED = Path('/tmp/uatd_processed' + ('_smoke' if SMOKE else ''))  # PNG copies; rebuilt per session, not in output

STATE.mkdir(parents=True, exist_ok=True)
def done(stage):  return (STATE / f'{stage}{"_smoke" if SMOKE else ""}.done').exists()
def mark(stage):  (STATE / f'{stage}{"_smoke" if SMOKE else ""}.done').write_text(time.ctime())
def sh(cmd):
    # stream the command's output into this cell so errors are visible
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait():
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}  (see the output above)')
def find_uatd_root(base='/kaggle/input', max_depth=5):
    # Kaggle may mount the dataset at different paths and may nest unzipped folders.
    for p in sorted(Path(base).rglob('UATD_Training')):
        if p.is_dir() and len(p.relative_to(base).parts) <= max_depth and (p.parent / 'UATD_Test_1').is_dir()                 and (p.parent / 'UATD_Test_2').is_dir():
            return str(p.parent)
    return None
RAW = RAW or find_uatd_root()
assert RAW, 'UATD not found under /kaggle/input: attach the dataset (Input -> + Add Input)'
print('UATD raw root:', RAW)

In [ ]:
# ---- restore from a previous version's output (optional) ----
if PREV_OUTPUT and not RUNS.exists():
    for sub in ('runs', 'state'):
        src = Path(PREV_OUTPUT) / sub
        if src.exists():
            shutil.copytree(src, WORK / sub, dirs_exist_ok=True); print('restored', src)

In [ ]:
# ---- setup: get the code (pull if already cloned) and install ----
if (CODE / '.git').exists():
    sh(f'git -C {CODE} fetch -q origin {BRANCH} && git -C {CODE} checkout -q {BRANCH} && git -C {CODE} pull -q --ff-only')
else:
    sh(f'git clone -q -b {BRANCH} {REPO} {CODE}')
os.chdir(CODE)
sh('git log -1 --format="%h %s"')
sh('pip install -q -e ".[dev,rfdetr]"')

In [ ]:
# ---- stage 1: verify the committed split against the raw data ----
if done('split_verified'):
    print('stage 1 already done')
else:
    sh(f'UATD_RAW_ROOT={RAW} python -m pytest -q -p no:cacheprovider tests/test_splits.py')
    mark('split_verified')

In [ ]:
# ---- stage 2: build YOLO/COCO views (lossless PNG) ----
if (PROCESSED / 'convert_report.json').exists():
    print('stage 2 already done for this session:', PROCESSED)
else:
    sh(f'python -m uatd.convert --raw-root {RAW} --out {PROCESSED} --link-mode png' + (' --smoke' if SMOKE else ''))

In [ ]:
# ---- stage 3: train RF-DETR (resumes automatically) ----
exp = f'{MODEL}_r{RESOLUTION or "default"}_s{SEED}' + ('_smoke' if SMOKE else '')
run_dir = RUNS / exp
status = json.loads((run_dir / 'status.json').read_text())['state'] if (run_dir / 'status.json').exists() else None
if status == 'done':
    print('stage 3 already done:', run_dir)
else:
    hours_left = TRAIN_HOURS - (time.time() - SESSION_START) / 3600
    args = (f'--data {PROCESSED} --model {MODEL} --epochs {EPOCHS} --batch {BATCH} --grad-accum {GRAD_ACCUM} '
            f'--seed {SEED} --runs-dir {RUNS} --exp {exp} --time-budget-hours {hours_left:.2f}')
    if RESOLUTION: args += f' --resolution {RESOLUTION}'
    if SMOKE:      args += ' --smoke'
    sh(f'python scripts/train_rfdetr.py {args}')
    status = json.loads((run_dir / 'status.json').read_text())['state']
    print('training status:', status)
    if status != 'done':
        print('Not finished yet. Re-run the notebook (or a new version with PREV_OUTPUT) to continue.')

In [ ]:
# ---- stage 4: predict + shared evaluator on val, test1, test2 ----
if status != 'done':
    print('skipping: training not finished')
else:
    sh(f'python scripts/predict_rfdetr.py --run {run_dir} --data {PROCESSED}')
    for split in ('val', 'test1', 'test2'):
        m = json.loads((run_dir / 'metrics' / f'{split}.json').read_text())
        print(f"{split:6s} mAP50 {m['mAP50']:.4f}  mAP75 {m['mAP75']:.4f}  mAP50:95 {m['mAP50_95']:.4f}")

In [ ]:
# ---- stage 5: small files to download / commit (no weights) ----
if status == 'done':
    out = WORK / f'results_{exp}'
    out.mkdir(exist_ok=True)
    for f in ['run_config.json', 'sessions.json', 'status.json']:
        if (run_dir / f).exists(): shutil.copy(run_dir / f, out / f)
    for d in ['metrics', 'predictions']:
        if (run_dir / d).exists(): shutil.copytree(run_dir / d, out / d, dirs_exist_ok=True)
    print('copy this folder into results/ in the repo:', out)